In [1]:
import os
import pandas

# Construction du csv emotion

In [2]:
FOLDER_EMOTION = "sentence_level_all_features"
FOLDER_EMOTION_UTF8 = "sentence_level_all_features_utf8"

In [3]:
# Conversion des fichiers MBCS -> UTF-8
os.makedirs(FOLDER_EMOTION_UTF8, exist_ok=True)

converted = 0
errors = []
for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    src_path = os.path.join(FOLDER_EMOTION, filename)
    dst_path = os.path.join(FOLDER_EMOTION_UTF8, filename)
    try:
        with open(src_path, "r", encoding="mbcs") as f:
            content = f.read()
        with open(dst_path, "w", encoding="utf-8") as f:
            f.write(content)
        converted += 1
    except Exception as e:
        errors.append((filename, str(e)))

print(f"{converted} fichier(s) converti(s) en UTF-8.")
if errors:
    print(f"{len(errors)} erreur(s) :", errors)

2896 fichier(s) converti(s) en UTF-8.


In [4]:


FOLDER_EMOTION = FOLDER_EMOTION_UTF8
features_found = set()

for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    filepath = os.path.join(FOLDER_EMOTION, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if "::" in line:
                feature_name = line.split("::")[0].strip()
                if feature_name and feature_name != "Features":
                    features_found.add(feature_name)

print(f"{len(features_found)} features uniques trouvées :\n")
for feat in sorted(features_found):
    print(f"  - {feat}")

16 features uniques trouvées :

  - avec_cat_admiration
  - avec_cat_autre
  - avec_cat_colere
  - avec_cat_embarras
  - avec_cat_fierte
  - avec_cat_joie
  - avec_cat_peur
  - avec_cat_surprise
  - avec_cat_tristesse
  - avec_emo
  - avec_emo_base
  - avec_emo_complexe
  - avec_emo_comportementale
  - avec_emo_designee
  - avec_emo_montree
  - avec_emo_suggeree


In [5]:
type(features_found)

set

In [6]:
NO_FEATURES_MESSAGE ="No features found."
ALL_FEATURES = list(features_found)

In [7]:
def extract_text(lines):
    if lines[-1]!=NO_FEATURES_MESSAGE:
     l = len(lines)
     trueline = []
     i = 4
     while True:
        if lines[i].startswith("Features::"):
            break
        trueline.append(lines[i])
        i+=1
        return " ".join(trueline).strip().split("Sentence: ")[1].strip()
    else:
     return " ".join(lines[4:-1]).strip().split("Sentence: ")[1].strip()


def extract_features(lines):
    features = {}
    for line in lines:
        line = line.strip()
        if "::" in line:
            parts = line.split("::")
            feature_name = parts[0].strip()
            if feature_name in ALL_FEATURES:
                features[feature_name] = 1
    return features


def extract_from_file(lines, id):

    row = {"file_id": id, "feature_found": False}
    for f in ALL_FEATURES:
        row[f] = 0

    if lines[-1].strip() == NO_FEATURES_MESSAGE:
        print("No features found for file id: ", id)
        row["text"] = ("\n").join(lines[3:-1]).split("Sentence: ")[1].strip()
    else:
        row["text"] = extract_text(lines)
        row["feature_found"] = True
        found = extract_features(lines)
        for fname, val in found.items():
            row[fname] = val

    return row

In [8]:
rows = []

for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    file_id = filename.replace(".txt", "")
    filepath = os.path.join(FOLDER_EMOTION, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().splitlines()
    # On filtre les lignes vides en fin de fichier
    while lines and lines[-1].strip() == "":
        lines.pop()
    rows.append(extract_from_file(lines, file_id))

No features found for file id:  100
No features found for file id:  1002
No features found for file id:  1003
No features found for file id:  1004
No features found for file id:  1005
No features found for file id:  1007
No features found for file id:  1008
No features found for file id:  1009
No features found for file id:  101
No features found for file id:  1010
No features found for file id:  1011
No features found for file id:  1012
No features found for file id:  1013
No features found for file id:  1014
No features found for file id:  1015
No features found for file id:  1016
No features found for file id:  1019
No features found for file id:  102
No features found for file id:  1020
No features found for file id:  1022
No features found for file id:  1024
No features found for file id:  1025
No features found for file id:  1026
No features found for file id:  1027
No features found for file id:  1028
No features found for file id:  1029
No features found for file id:  1030
No f

In [9]:
print(len(rows), "rows extracted.")
df = pandas.DataFrame(rows)

2896 rows extracted.


In [10]:
df = df.set_index("file_id")
df.index = pandas.to_numeric(df.index, errors='coerce')
df = df.sort_index()
df.head()

,feature_found,avec_emo_comportementale,avec_cat_fierte,avec_emo_complexe,avec_cat_colere,avec_cat_joie,avec_cat_autre,avec_cat_tristesse,avec_emo,avec_cat_surprise,avec_emo_designee,avec_emo_base,avec_cat_peur,avec_cat_admiration,avec_emo_montree,avec_emo_suggeree,avec_cat_embarras,text
file_id,,,,,,,,,,,,,,,,,,
1,True,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,"C'est dingue, après tout ce temps, plus de 15 ..."
2,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"Plutôt d'accord pour les SAV, c'est le but mêm..."
3,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Bordel.
4,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,J'avais tapé une longue réponse.
5,False,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"Une fausse manip sur le téléphone, et hop, per..."


In [11]:
df['feature_found'].value_counts()

feature_found
False    2183
True      713
Name: count, dtype: int64

In [12]:
# Aperçu des features binaires pour les phrases avec au moins une feature
feature_cols = ALL_FEATURES
df[df['feature_found'] == True][['text'] + feature_cols].head(10)

,text,avec_emo_comportementale,avec_cat_fierte,avec_emo_complexe,avec_cat_colere,avec_cat_joie,avec_cat_autre,avec_cat_tristesse,avec_emo,avec_cat_surprise,avec_emo_designee,avec_emo_base,avec_cat_peur,avec_cat_admiration,avec_emo_montree,avec_emo_suggeree,avec_cat_embarras
file_id,,,,,,,,,,,,,,,,,
1,"C'est dingue, après tout ce temps, plus de 15 ...",1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0
7,Non pas de problème de ce côté là heureusement !,0,0,0,0,1,0,0,1,0,1,1,0,0,0,0,0
8,Les violences étaient clairement motivées par ...,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
10,"A l'époque, si je comprends bien, tu t'identif...",0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
12,Je vois donc comment tu peux penser avoir reçu...,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
13,En quoi c'est putaclic ?,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
17,La colonisation et l'esclavage domestique fond...,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
22,La colonisation a accentué l'appauvrissement e...,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1,0
30,Donc j'espère vraiment que ton médecin a décla...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [14]:
df.to_csv("corpus_emotyc_reddit_features.csv", index=True)